In [30]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph,START,END
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from dotenv import load_dotenv
from typing import TypedDict,Literal,Dict,List,Annotated,Optional
from pydantic import BaseModel,Field
load_dotenv()
model=ChatOpenAI()


python-dotenv could not parse statement starting at line 5


In [31]:
class ReviewState(TypedDict):
    review:str
    Diagnosis:dict
    sentiment:Literal['positive','negative']
    response:str

class SentimentSchema(BaseModel):
    sentiment:Literal['postive','negative']=Field(description="generate the sentiment for the given inptu")
structured_model=model.with_structured_output(SentimentSchema)

class DiagnosisSchema(BaseModel):
    IssueType:Literal['UI','Perfomance','Bug','Support']=Field(description="Issue type")
    Tone:Literal['Angry','Calm','Frustrated']=Field(description="Tone")
    urgency:Literal['low','medium','high']=Field(description="Type of urgency")
structured_model2=model.with_structured_output(DiagnosisSchema)

d:\VSCode_AgenticAI\myenv\Lib\site-packages\langchain_openai\chat_models\base.py:2493: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [32]:
prompt1=PromptTemplate(
        template="generate the positive AI response for the input:{Input}",
        input_variables=['Input']
    )
prompt2=PromptTemplate(
        template="generate the negative AI response for the input:{Input}",
        input_variables=['Input']
    )
parser=StrOutputParser()

def Find_sentiment(state:ReviewState):
    review=state['review']
    result=structured_model.invoke(review)
    return {'sentiment':result.sentiment}


def positive_response(state:ReviewState):
   sentiment=state['sentiment']
   chain=RunnableSequence(prompt1,model,parser)
   result=chain.invoke({'Input':sentiment})
   return {'response',result}

def negative_response(state:ReviewState):
   sentiment=state['sentiment']
   chain=RunnableSequence(prompt1,model,parser)
   result=chain.invoke({'Input':sentiment})
   return {'response',result}

def run_diagnosis(state:ReviewState):
   review=state['review']
   prompt3=f'given input review:{review}, could you please generate the issueType,tone and urgency'
   result=structured_model2.invoke(prompt3)
   return {'Dianosis':result.model_dump()}

def Check_condition(state:ReviewState)->Literal['run_diagnosis','positive_response']:
   if state['sentiment']=='positive':
      return "positive_response"
   else:
      return 'run_diagnosis'
   
graph=StateGraph(ReviewState)
graph.add_node('Find_sentiment',Find_sentiment)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('positive_response',positive_response)
graph.add_node('negative_response',negative_response)
graph.add_edge(START,'Find_sentiment')
graph.add_conditional_edges('Find_sentiment',Check_condition)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('positive_response',END)
graph.add_edge('negative_response',END)
graph.compile()

workflow=graph.compile()
initial_state={'review':'I didnot  like this mobile,display is not working and i need to return this immediately'}
final_state=workflow.invoke(initial_state)
print(final_state)


InvalidUpdateError: Expected dict, got {'response', "I'm sorry to hear that you're feeling negative. Is there anything specific that is bothering you? Let's try to focus on finding a solution or looking at things from a different perspective to see if we can turn it into a positive experience. Remember, tough times don't last, but tough people do. You are strong and capable of overcoming any challenges that come your way."}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE